# PLS: 10 m cells vs 100 m mean-pooled tiles

LORO over the same 10 metros. Features are 64-D AlphaEarth at both scales — at 100 m each tile is the **mean** of its 10 m cell embeddings (not the 10×10 grid model).

| | 10 m | 100 m |
|---|---:|---:|
| Units | 1,751,814 pure cells | 52,492 tiles (min 5 cells) |
| Balanced RES / NRES | 875,907 / 875,907 | 26,246 / 26,246 |
| Best PLS-DA F1 | 0.775 (k=32) | 0.762 (k=24) |
| 64-D LR F1 | 0.779 | 0.761 |
| t₁-only F1 | 0.735 | 0.720 |

**Takeaway.** Mean-pooling to 100 m costs about **0.013** macro-F1 on the best PLS-DA head. The sparsity story is the same at both scales: strong L1 (C=10) leaves ~1 nonzero score ($t_1$), with F1 matching the explicit $t_1$-only head.

```bash
.venv/bin/python -u scripts/train_week4_pls_10m_vs_100m.py --backend gpu
.venv/bin/python -u scripts/build_week4_pls_10m_vs_100m_notebook.py
```


## Result table

LORO mean macro-F1. Δ = 10 m − 100 m.

| Head | 10 m | 100 m | Δ |
|---|---:|---:|---:|
| 64-D LR | 0.779 | 0.761 | 0.018 |
| PLS-DA best-k | 0.775 | 0.762 | 0.013 |
| t₁ only | 0.735 | 0.720 | 0.014 |
| L1 on PLS (C=10) | 0.735 | 0.721 | 0.014 |
| L1 on PLS (C=30) | 0.751 | 0.738 | 0.013 |
| PCA best-k | 0.759 | 0.749 | 0.010 |

Pooled OOF F1 (all metros concatenated): 10 m baseline 0.805, PLS 0.804; 100 m baseline 0.787, PLS 0.790.


## F1 vs number of PLS components

![F1 vs k](../../outputs/week4_pls_10m_vs_100m/figures/f1_vs_k_by_scale.png)

At $k=1$: 10 m F1 ≈ 0.735, 100 m ≈ 0.720. Best 10 m at k=32; best 100 m at k=24.

## Heads side by side

![Family compare](../../outputs/week4_pls_10m_vs_100m/figures/f1_family_10m_vs_100m.png)


## Per-metro gap

![Delta by metro](../../outputs/week4_pls_10m_vs_100m/figures/delta_f1_by_metro.png)

Mean metro Δ (best PLS-DA) ≈ 0.013. 10 m is higher in every fold.


## L1 sparsity (same pattern at both scales)

![L1 nnz](../../outputs/week4_pls_10m_vs_100m/figures/l1_nnz_by_scale.png)

| Scale | C=10 mean nnz | C=10 F1 | t₁-only F1 | C=30 mean nnz | C=30 F1 |
|---|---:|---:|---:|---:|---:|
| 10 m | 1.2 | 0.735 | 0.735 | 6.3 | 0.751 |
| 100 m | 1.1 | 0.721 | 0.720 | 6.9 | 0.738 |

$C=10$ ≈ $t_1$ only. $C=30$ keeps ~6–7 scores and picks up ~0.015–0.016 F1 at both scales.


## Fraction MAE

![MAE vs k](../../outputs/week4_pls_10m_vs_100m/figures/mae_vs_k_by_scale.png)

Best PLS fraction MAE: 10 m 0.260 (k=12), Ridge 0.263.  
100 m 0.139 (k=12), Ridge 0.141.

MAE is not a clean scale comparison: 10 m uses pure balanced cells (often extreme fractions), 100 m uses tile-mean fractions over mixed cells. Treat F1 as the primary scale metric.


## What this does *not* compare

- **10×10 grid LR** on 100 m tiles (`outputs/week4_100m_tile_train/`) — 6500-D spatial input, different feature space.
- **Pooling 10 m predictions** onto tiles (`outputs/week4_100m_fraction_eval/`) — aggregation of cell scores, not a new PLS fit.

Here the question is: if you keep a single 64-D AE vector and only change the spatial unit (cell vs mean-pooled tile), how does PLS / $t_1$ / L1 behave?


## Artifacts

| Path | Contents |
|---|---|
| `outputs/week4_pls_10m_vs_100m/summary.json` | Cross-scale numbers |
| `outputs/week4_pls_10m_vs_100m/sweep_summary.csv` | F1 / MAE / nnz by scale × model × k |
| `outputs/week4_pls_10m_vs_100m/10m/` · `100m/` | Per-scale OOF, fold checkpoints, L1 coefs |
| `outputs/week4_pls_10m_vs_100m/figures/` | Claim-first plots |

Elapsed ≈ 238.7 s on 8 GPUs.
